# Hierarchical process (with Tools call)

Agents orchestration with a manager agent.

# Installation

In [15]:
!pip install -q crewai crewai_tools

# Import Dependencies

In [16]:
import crewai
import crewai_tools

print(crewai.__version__)
print(crewai_tools.__version__)

1.15.10
1.15.10


In [10]:
# Import dependencies
import os
from crewai import Agent, Task, Crew, LLM, Process
from crewai_tools import SerperDevTool, DirectoryReadTool

# If loading from a .env file
# from dotenv import load_dotenv
# load_dotenv()


# Set API Keys

In [3]:
from google.colab import userdata
import os
os.environ["SERPER_API_KEY"] = userdata.get('SERPER_API_KEY')
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY_NEW')
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')


# Create LLM Objects

In [ ]:
# -----------
# Create LLM
# -----------

# Create an LLM with a temperature of 0 to ensure deterministic outputs
# OPENAI LLMs
manager_llm = LLM(
         # model="gpt-5.4-mini",
          model="gpt-5.4-nano",
          base_url="https://api.openai.com/v1",
          api_key = os.environ["OPENAI_API_KEY"],
          temperature=0.2)

# GROQ hosted LLMs
llm = LLM(
     model="llama-3.3-70b-versatile",
     base_url="https://api.groq.com/openai/v1",
     api_key=os.environ["GROQ_API_KEY"],
     temperature=0.7)


# Tools

In [6]:
#------------------------------------------------------------------------
# Instantiate the Tools
#------------------------------------------------------------------------

search_tool = SerperDevTool()

docs_tool = DirectoryReadTool(directory='./blog-posts')

# Agents

In [11]:
# --------------
# Define Agents
# --------------
manager_agent = Agent(
    role="Content Manager",
    goal="Coordinate the blog creation process by assigning subtasks to the research specialist and content writer agents.",
    backstory=(
        "You are an experienced content manager. "
        "You decide the order of work, delegate to other coworker agents, and ensure the final task completion."
        "You do NOT solve the problem on your own. You need to coordinate amongst the coworker agents to get the task completed."
        "Ideally you should kick off the researcher first to collect information and then pass the research findings to the content writer agent."
        "Coworker agents may have access to specific tools that they should use to complete their tasks."
    ),
    llm=manager_llm, # Make sure to use a more capable LLM here
    max_iter=3,
    allow_delegation=True,
    verbose=True
)

research_agent = Agent(
    role="Research Specialist",
    goal="Gather accurate and relevant information in real-time for a given topic.",
    backstory="You are an expert in web research, skilled at finding key facts, statistics, and trends.",
    llm=llm,
    max_iter=3,
    tools=[search_tool],
    allow_delegation=False,
    verbose=True
)

writer_agent = Agent(
    role="Content Writer",
    goal="Produce high-quality blog articles from the provided research material.",
    backstory="You write clear, engaging, and well-structured content.",
    llm=llm,
    tools=[docs_tool],
    max_iter=3,
    max_rpm=15,
    allow_delegation=False,
    verbose=True
)


# Tasks

In [12]:
# --------------------
# Define Tasks
# --------------------
research_task = Task(
    description=(
        "Research the topic '{topic}'. Provide 5-7 bullet points "
        "with key facts, statistics, and relevant examples."
        "Use the provided tool to search for the latest information about the topic. "
        "Do NOT rely on memory — you must call the search tool at least once."
    ),
    expected_output="A bullet-point list of factual research notes.",
    agent=research_agent,
    verbose=True

)

writing_task = Task(
    description=(
        "Write a 500-word blog post on '{topic}' using this research done by the researcher."
    ),
    expected_output="A fully written blog post.",
    agent=writer_agent,

    # Let the manager pass the context as required.
    # You do not need to set any order of execution of the agents/tasks or pass the context
    # DO NOT SET THE FOLLOWING ATTRIBUTES
    # depends_on=[research_task],
    # context = [research_task], # This passes the context of the research task as a Task object not as strings as expected by the writin agent
                                 # Comment this line and just use depends_on to pass the context as a string

    output_file="blog-posts/new_tech_post_{topic}.md",  # The final blog post will be saved here
    verbose=True
)


# Crew (Orchestration Layer)

In [13]:
# -------------------------------------
# Define Crew with Hierarchical Process
# -------------------------------------
crew = Crew(
    agents=[research_agent, writer_agent], # We do NOT specify the manager agent in this list
    tasks=[research_task, writing_task],
    process=Process.hierarchical,    # <--- THIS sets the hierarchical workflow
    manager_agent=manager_agent,     # <--- EITHER Explicitly set the manager agent
    # manager_llm="openai/gpt-4o", # <--- OR Explicitly set the manager LLM, make sure to use a more capable LLM here
    planning=True,
    verbose=True
)


In [14]:
# -------------
# Run the Crew
# -------------
result = await crew.kickoff_async(inputs={"topic": "The future of job market for CS undergrad in the era of Gen AI"})
# result = await crew.kickoff_async(inputs={"topic": "The future of Humanity as we tend to achieve AGI"})
# result = await crew.kickoff_async(inputs={"topic": "About OpenAI's latest GPT-5 model. Major technical heighlights as 5 bullet points. Additional 3 bullet points highlighting how it is better than GPT-4"})

print("\n=== FINAL OUTPUT ===")
print(result)


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 5bd5c71d-a6af-494e-a7a3-18baf261d5aa                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


[2026-08-03 04:51:57][INFO]: Planning the crew execution


╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Based on these tasks summary:                                                                            │
│                  Task Number 1 - Research the topic 'The future of job market for CS undergrad in the era of    │
│  Gen AI'. Provide 5-7 bullet points with key facts, statistics, and relevant examples.Use the provided tool to  │
│  search for the latest information about the topic. Do NOT rely on memory — you must call the search tool at    │
│  least once.                                                                                                    │
│                  "task_description": Research the topic 'The future of job market for CS undergrad in the era   │
│  of Gen AI'. Provide 5-7 bullet points with key facts, statistics, and relevant examples.Use the provided tool  │
│  to search for the latest information about the topic. Do NOT rely on memory — you must call the search tool    │
│  at least once.                                                                                                 │
│                  "task_expected_output": A bullet-point list of factual research notes.                         │
│                  "agent": Research Specialist                                                                   │
│                  "agent_goal": Gather accurate and relevant information in real-time for a given topic.         │
│                  "task_tools": [SerperDevTool(name='Search the internet with Serper', description="A tool that  │
│  can be used to search the internet with a search_query. Supports different search types: 'search' (default),   │
│  'news'", env_vars=[EnvVar(name='SERPER_API_KEY', description='API key for Serper', required=True,              │
│  default=None)], args_schema=<class 'crewai_tools.tools.serper_dev_tool.serper_dev_tool.SerperDevToolSchema'>,  │
│  result_schema=None, description_updated=False, cache_function=<function _default_cache_function at             │
│  0x783aa861aac0>, result_as_answer=False, max_usage_count=None, tool_failure_policy=None,                       │
│  current_usage_count=0, base_url='https://google.serper.dev', n_results=10, save_file=False,                    │
│  search_type='search', country='', location='', locale='',                                                      │
│  tool_type='crewai_tools.tools.serper_dev_tool.serper_dev_tool.SerperDevTool')]                                 │
│                  "agent_tools": [name='Search the internet with Serper' description="A tool that can be used    │
│  to search the internet with a search_query. Supports different search types: 'search' (default), 'news'"       │
│  env_vars=[EnvVar(name='SERPER_API_KEY', description='API key for Serper', required=True, default=None)]        │
│  args_schema=<class 'crewai_tools.tools.serper_dev_tool.serper_dev_tool.SerperDevToolSchema'>                   │
│  result_schema=None description_updated=False cache_function=<function _default_cache_function at               │
│  0x783aa861aac0> result_as_answer=False max_usage_count=None tool_failure_policy=None current_usage_count=0     │
│  base_url='https://google.serper.dev' n_results=10 save_file=False search_type='search' country='' location=''  │
│  locale='' tool_type='crewai_tools.tools.serper_dev_tool.serper_dev_tool.SerperDevTool']                        │
│                  Task Number 2 - Write a 500-word blog post on 'The future of job market for CS undergrad in    │
│  the era of Gen AI' using this research done by the researcher.                                                 │
│                  "task_description": Write a 500-word b

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Based on these tasks summary:                                                                            │
│                  Task Number 1 - Research the topic 'The future of job market for CS undergrad in the era of    │
│  Gen AI'. Provide 5-7 bullet points with key facts, statistics, and relevant examples.Use the provided tool to  │
│  search for the latest information about the topic. Do NOT rely on memory — you must call the search tool at    │
│  least once.                                                                                                    │
│                  "task_description": Research the topic 'The future of job market for CS undergrad in the era   │
│  of Gen AI'. Provide 5-7 bullet points with key facts, statistics, and relevant examples.Use the provided tool  │
│  to search for the latest information about the topic. Do NOT rely on memory — you must call the search tool    │
│  at least once.                                                                                                 │
│                  "task_expected_output": A bullet-point list of factual research notes.                         │
│                  "agent": Research Specialist                                                                   │
│                  "agent_goal": Gather accurate and relevant information in real-time for a given topic.         │
│                  "task_tools": [SerperDevTool(name='Search the internet with Serper', description="A tool that  │
│  can be used to search the internet with a search_query. Supports different search types: 'search' (default),   │
│  'news'", env_vars=[EnvVar(name='SERPER_API_KEY', description='API key for Serper', required=True,              │
│  default=None)], args_schema=<class 'crewai_tools.tools.serper_dev_tool.serper_dev_tool.SerperDevToolSchema'>,  │
│  result_schema=None, description_updated=False, cache_function=<function _default_cache_function at             │
│  0x783aa861aac0>, result_as_answer=False, max_usage_count=None, tool_failure_policy=None,                       │
│  current_usage_count=0, base_url='https://google.serper.dev', n_results=10, save_file=False,                    │
│  search_type='search', country='', location='', locale='',                                                      │
│  tool_type='crewai_tools.tools.serper_dev_tool.serper_dev_tool.SerperDevTool')]                                 │
│                  "agent_tools": [name='Search the internet with Serper' description="A tool that can be used    │
│  to search the internet with a search_query. Supports different search types: 'search' (default), 'news'"       │
│  env_vars=[EnvVar(name='SERPER_API_KEY', description='API key for Serper', required=True, default=None)]        │
│  args_schema=<class 'crewai_tools.tools.serper_dev_tool.serper_dev_tool.SerperDevToolSchema'>                   │
│  result_schema=None description_updated=False cache_function=<function _default_cache_function at               │
│  0x783aa861aac0> result_as_answer=False max_usage_count=None tool_failure_policy=None current_usage_count=0     │
│  base_url='https://google.serper.dev' n_results=10 save_file=False search_type='search' country='' location=''  │
│  locale='' tool_type='crewai_tools.tools.serper_dev_tool.serper_dev_tool.SerperDevTool']                        │
│                  Task Number 2 - Write a 500-word blog post on 'The future of job market for CS undergrad in    │
│  the era of Gen AI' using this research done by the researcher.                                                 │
│                  "task_description": Write a 500-word b

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Research the topic 'The future of job market for CS undergrad in the era of Gen AI'. Provide 5-7 bullet  │
│  points with key facts, statistics, and relevant examples.Use the provided tool to search for the latest        │
│  information about the topic. Do NOT rely on memory — you must call the search tool at least once.1. Start by   │
│  clarifying the research objective: produce 5-7 concise bullet points that are factual, current, and directly   │
│  relevant to the future job market for computer science undergraduates in the era of generative AI.             │
│  2. Use the Serper internet search tool at least once, and preferably multiple times, with highly targeted      │
│  queries such as:                                                                                               │
│     - "future of CS jobs Gen AI 2024 2025"                                                                      │
│     - "generative AI impact on software engineering jobs statistics"                                            │
│     - "computer science graduates job market AI hiring trends"                                                  │
│     - "tech hiring outlook AI entry-level jobs"                                                                 │
│     - "WEF future of jobs report AI software developer"                                                         │
│  3. Prioritize recent sources from reputable outlets and institutions, including labor-market reports, tech     │
│  industry analyses, university career reports, consulting firms, and major news coverage. Favor sources with    │
│  dates, quantitative findings, and explicit mentions of entry-level or CS-related roles.                        │
│  4. During search result review, capture concrete evidence such as:                                             │
│     - percentage changes in hiring or layoffs                                                                   │
│     - projections for roles likely to grow or shrink                                                            │
│     - employer expectations for AI literacy, coding assistance tools, and automation                            │
│     - examples of job titles affected, such as software engineer, QA tester, data analyst, or prompt/AI         │
│  engineer                                                                                                       │
│     - any reported shifts in internship or junior-level opportunities                                           │
│  5. Cross-check any important statistic against more than one source when possible. If sources conflict, note   │
│  the discrepancy and prefer the most recent or most authoritative source.                                       │
│  6. Extract 5-7 bullet points that each include one clear fact, one statistic or specific example, and a brief  │
│  implication for CS undergrads. Keep each bullet focused and evidence-based.                                    │
│  7. Ensure the notes reflect the current landscape rather than generic long-term opinions: include what Gen AI  │
│  is changing now in hiring, skill requirements, productivity expectations, and competition for entry-level      │
│  roles.                                                                                                         │
│  8. If available, include examples of how CS undergrads can differentiate themselves in the market, such as     │
│  building AI-assisted projects, understanding applied ML, using coding copilots responsibly, or showcasing      │
│  domain-specific problem solving—only if supported by t

...

Maximum iterations reached. Requesting final answer.
[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Content Manager                                                                                         │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ## The future of the job market for CS undergrads in the era of Gen AI                                         │
│                                                                                                                 │
│  Gen AI is changing software work fast—and the CS job market is feeling it. If you’re a CS undergrad (or        │
│  advising one), the key shift isn’t that “AI replaces programmers.” It’s that AI is reshaping *which parts* of  │
│  entry-level work are needed, *how* teams hire, and *what proof of skill* employers expect from new grads.      │
│                                                                                                                 │
│  ### 1) Entry-level roles are being reshaped (and “pure junior” openings may shrink)                            │
│  One clear signal: after widespread GenAI adoption, a study on generative AI’s impact on software engineering   │
│  found a **16.3% drop in the relative proportion of junior vs. senior-level software roles**. In plain terms,   │
│  companies may still hire, but they’re less likely to staff large numbers of straightforward “junior-only”      │
│  tasks. That doesn’t mean you can’t get hired—it means you’ll likely compete for roles that require faster      │
│  ownership and clearer judgment.                                                                                │
│                                                                                                                 │
│  At the same time, hiring for entry-level tech roles has been under pressure. CNBC, citing Revelio Labs labor   │
│  research, reported **entry-level job postings in the U.S. declined about 35% since January 2023**. So while    │
│  AI can create new demand, it can also concentrate hiring around fewer candidates who can ramp quickly.         │
│                                                                                                                 │
│  **What this means for you:** aim for internships and projects that show end-to-end delivery—requirements,      │
│  implementation, testing, and deployment—not just “ticket execution.”                                           │
│                                                                                                                 │
│  ### 2) Skill expectations are shifting: AI literacy is becoming baseline                                       │
│  Employers increasingly treat GenAI skills as table stakes. The National Association of Colleges and Employers  │
│  (NACE) reports **more than one-third of entry-level jobs require AI skills**, described as **nearly triple**   │
│  compared with Fall 2025. That’s a big change in what “entry-level” means.                                      │
│                                                                                                                 │
│  There’s also evidence that AI exposure can affect early-career outcomes. Reporting based on a Stanford         │
│  Digital Economy study indicates that for **jobs with the most AI exposure (including IT/software               │
│  engineering), employment for workers aged 22–25 declined nearly 20%**. The takeaway isn’t “avoid AI.” It’s     │
│  that early-career workers may need to differentiate faster—by working effectively with AI rather than relying  │
│  on it.                                                

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Write a 500-word blog post on 'The future of job market for CS undergrad in the era of Gen AI' using     │
│  this research done by the researcher.1. First, use the DirectoryReadTool to inspect the ./blog-posts           │
│  directory and identify any existing files, templates, prior drafts, or style references that may inform tone,  │
│  structure, formatting, and preferred voice for the blog post.                                                  │
│  2. Review the researcher’s notes carefully and extract the strongest arguments, statistics, and examples that  │
│  can be woven into a coherent narrative. Prioritize the most current and relevant evidence rather than trying   │
│  to include every detail.                                                                                       │
│  3. Define a clear blog structure before drafting:                                                              │
│     - attention-grabbing introduction about Gen AI reshaping the CS job market                                  │
│     - 2-3 body paragraphs explaining impact on entry-level hiring, changing skill expectations, and             │
│  opportunities for CS undergrads                                                                                │
│     - a concluding paragraph with practical advice and an optimistic but realistic outlook                      │
│  4. Write in a polished, readable blog style suitable for a general audience, avoiding academic jargon while    │
│  still preserving factual accuracy. Maintain a balanced tone: acknowledge risks such as automation and          │
│  competition, but also highlight new opportunities created by AI adoption.                                      │
│  5. Integrate the research facts naturally into the prose. Use statistics and examples to support claims, but   │
│  do not overload the article with numbers. Each statistic should serve a clear point about the changing job     │
│  landscape.                                                                                                     │
│  6. Emphasize actionable implications for CS undergrads, such as:                                               │
│     - building AI-fluent programming skills                                                                     │
│     - learning to work alongside AI tools                                                                       │
│     - strengthening fundamentals in algorithms, systems, and problem solving                                    │
│     - developing projects that demonstrate real-world application                                               │
│     Only include recommendations that are consistent with the research findings.                                │
│  7. Keep the post around 500 words by controlling paragraph length and focusing on the most important           │
│  insights. Avoid unnecessary repetition and ensure smooth transitions between ideas.                            │
│  8. Use the directory contents to match any existing stylistic conventions, such as heading usage, formatting,  │
│  or call-to-action patterns, if such examples exist.                                                            │
│  9. After drafting, edit for clarity, flow, and accuracy. Confirm that the final blog post is self-contained,   │
│  compelling, and aligned with the research-based message.                                                       │
│  10. Return a fully written blog post as the final output, with no placeholder text, and ensure the content     │
│  reflects the current research rather than unsupported 


=== FINAL OUTPUT ===
## The future of the job market for CS undergrads in the era of Gen AI

Gen AI is changing software work fast—and the CS job market is feeling it. If you’re a CS undergrad (or advising one), the key shift isn’t that “AI replaces programmers.” It’s that AI is reshaping *which parts* of entry-level work are needed, *how* teams hire, and *what proof of skill* employers expect from new grads.

### 1) Entry-level roles are being reshaped (and “pure junior” openings may shrink)
One clear signal: after widespread GenAI adoption, a study on generative AI’s impact on software engineering found a **16.3% drop in the relative proportion of junior vs. senior-level software roles**. In plain terms, companies may still hire, but they’re less likely to staff large numbers of straightforward “junior-only” tasks. That doesn’t mean you can’t get hired—it means you’ll likely compete for roles that require faster ownership and clearer judgment.

At the same time, hiring for entry-lev

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯